# Subproblem 3: Customer Advocacy and Retention Prioritisation

This notebook compares Promoters, Passives and Detractors by financial contribution, client profile and observed longevity. It uses exactly three charts: one Plotly Express chart and two Plotly Graph Objects charts. The charts are descriptive: without churn or renewal outcomes, retention priority means commercially important accounts that merit attention, not proven churn risk.


## Definitions and grain

- Source grain: one client-year record.
- Client grain: one row per client for retention decisions.
- Gross profit = Revenue − Hardware − Software − Manpower.
- Gross margin = Gross profit ÷ Revenue.
- NPS category: Promoter = 9–10, Passive = 7–8, Detractor = 0–6.
- Observed longevity = last observed year − first observed year + 1. This is observed relationship span, not guaranteed contract tenure.
- Priority quadrants use the client-level medians for satisfaction and gross margin; they are relative to this dataset.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

DATA_PATH = Path('merged_cleaned.xlsx')
df = pd.read_excel(DATA_PATH)
required = {'CLIENT ID','YEAR','REVENUE','HARDWARE','SOFTWARE','MANPOWER','NPS RATING',
            'PRESALES AND PARTNERSHIP','TECHNICAL EXPERTISE','PROJECT DELIVERY',
            'POST-SALES SUPPORT','TYPE','SECTOR','STAFF STRENGTH','COUNTRY'}
missing = sorted(required - set(df.columns))
if missing: raise KeyError(missing)
print(f'Loaded {len(df):,} client-year rows; {df["CLIENT ID"].nunique():,} unique clients')
print('Duplicate client-year rows:', int(df.duplicated(['CLIENT ID','YEAR']).sum()))

Loaded 392 client-year rows; 140 unique clients
Duplicate client-year rows: 0


In [2]:
analysis = df.copy()
service_cols = ['PRESALES AND PARTNERSHIP','TECHNICAL EXPERTISE','PROJECT DELIVERY','POST-SALES SUPPORT']
analysis['Overall Satisfaction'] = analysis[service_cols].mean(axis=1)
analysis['COGS'] = analysis[['HARDWARE','SOFTWARE','MANPOWER']].sum(axis=1, min_count=1)
analysis['Gross Profit'] = analysis['REVENUE'] - analysis['COGS']
analysis['Gross Margin'] = analysis['Gross Profit'] / analysis['REVENUE'].replace(0, np.nan)
analysis['NPS Category'] = np.select([analysis['NPS RATING'].ge(9), analysis['NPS RATING'].ge(7)], ['Promoter','Passive'], default='Detractor')
category_order = ['Promoter','Passive','Detractor']
analysis['NPS Category'] = pd.Categorical(analysis['NPS Category'], categories=category_order, ordered=True)

client = (analysis.groupby('CLIENT ID', as_index=False, observed=True).agg(
    Client_Type=('TYPE','first'), Sector=('SECTOR','first'), Organisation_Size=('STAFF STRENGTH','first'), Country=('COUNTRY','first'),
    Revenue=('REVENUE','sum'), Gross_Profit=('Gross Profit','sum'), Average_Satisfaction=('Overall Satisfaction','mean'),
    Average_NPS=('NPS RATING','mean'), First_Observed_Year=('YEAR','min'), Last_Observed_Year=('YEAR','max'),
    Observed_Years=('YEAR','nunique')).rename(columns={'CLIENT ID':'Client ID'}))
client['Gross_Margin'] = client['Gross_Profit'] / client['Revenue'].replace(0, np.nan)
client['Observed_Longevity'] = client['Last_Observed_Year'] - client['First_Observed_Year'] + 1
client['NPS Category'] = pd.Categorical(np.select([client['Average_NPS'].ge(9), client['Average_NPS'].ge(7)], ['Promoter','Passive'], default='Detractor'), categories=category_order, ordered=True)
client['Priority Group'] = np.select(
    [(client['Average_Satisfaction'] < client['Average_Satisfaction'].median()) & (client['Gross_Margin'] >= client['Gross_Margin'].median()),
     (client['Average_Satisfaction'] >= client['Average_Satisfaction'].median()) & (client['Gross_Margin'] >= client['Gross_Margin'].median()),
     (client['Average_Satisfaction'] >= client['Average_Satisfaction'].median()) & (client['Gross_Margin'] < client['Gross_Margin'].median())],
    ['Prioritise','Retain','Improve'], default='Reconsider')

summary = client.groupby('NPS Category', observed=True).agg(Clients=('Client ID','size'), Revenue=('Revenue','sum'), Gross_Profit=('Gross_Profit','sum'), Median_Longevity=('Observed_Longevity','median')).reindex(category_order).reset_index()
summary['Gross_Margin'] = summary['Gross_Profit'] / summary['Revenue']
display(summary.style.format({'Revenue':'${:,.0f}','Gross_Profit':'${:,.0f}','Gross_Margin':'{:.1%}','Median_Longevity':'{:.1f}'}))

,NPS Category,Clients,Revenue,Gross_Profit,Median_Longevity,Gross_Margin
0,Promoter,29,"$24,509,544","$10,816,841",1.0,44.1%
1,Passive,91,"$99,280,791","$43,844,747",3.0,44.2%
2,Detractor,20,"$11,290,903","$5,027,601",2.0,44.5%


## Chart 1 of 3 — Gross-profit contribution by advocacy group (Plotly Express)

This ranked horizontal bar chart shows where gross profit is concentrated across Promoters, Passives and Detractors. The dropdown changes the client-profile breakdown. Gross profit is the default metric because it directly measures financial contribution; hover details also show revenue, margin, client count, longevity and satisfaction.


In [3]:
profile_columns = {
    'Client type': 'Client_Type',
    'Sector': 'Sector',
    'Organisation size': 'Organisation_Size',
    'Country': 'Country'
}

# Build one Plotly Express view for each profile breakdown, then combine them
# into one figure so the dropdown can switch between profile dimensions.
fig1 = None
trace_counts = {}
for profile_label, profile_col in profile_columns.items():
    bars = (client.groupby([profile_col, 'NPS Category'], dropna=False, observed=True)
            .agg(Gross_Profit=('Gross_Profit', 'sum'), Revenue=('Revenue', 'sum'),
                 Clients=('Client ID', 'size'),
                 Median_Longevity=('Observed_Longevity', 'median'),
                 Average_Satisfaction=('Average_Satisfaction', 'mean'))
            .reset_index()
            .rename(columns={profile_col: 'Profile'}))
    bars['Profile'] = bars['Profile'].fillna('Unknown').astype(str)
    bars['Gross_Margin'] = bars['Gross_Profit'] / bars['Revenue'].replace(0, np.nan)
    bars['Gross_Profit_Share'] = bars['Gross_Profit'] / bars['Gross_Profit'].sum()
    bars['NPS Category'] = pd.Categorical(bars['NPS Category'], categories=category_order, ordered=True)
    bars = bars.sort_values(['Gross_Profit', 'NPS Category'], ascending=[True, True])
    bars['Bar_Label'] = bars['Profile'] + ' — ' + bars['NPS Category'].astype(str)
    bars['Text'] = bars['Gross_Profit'].map(lambda x: f'${x/1e6:.1f}M')

    px_view = px.bar(
        bars, x='Gross_Profit', y='Bar_Label', color='NPS Category', orientation='h',
        category_orders={'NPS Category': category_order},
        color_discrete_map={'Promoter': '#168aad', 'Passive': '#f4a261', 'Detractor': '#e76f51'},
        text='Text', custom_data=['Profile', 'NPS Category', 'Revenue', 'Gross_Profit',
                                  'Gross_Margin', 'Clients', 'Median_Longevity',
                                  'Average_Satisfaction', 'Gross_Profit_Share'],
        template='plotly_white')
    px_view.update_traces(
        hovertemplate='<b>%{customdata[0]} — %{customdata[1]}</b><br>'
                      'Gross profit: $%{customdata[3]:,.0f}<br>'
                      'Share of all gross profit: %{customdata[8]:.1%}<br>'
                      'Revenue: $%{customdata[2]:,.0f}<br>'
                      'Gross margin: %{customdata[4]:.1%}<br>'
                      'Clients: %{customdata[5]}<br>'
                      'Median observed longevity: %{customdata[6]:.1f} years<br>'
                      'Average satisfaction: %{customdata[7]:.2f}/5<extra></extra>',
        textposition='outside')
    if fig1 is None:
        fig1 = px_view
        fig1.update_layout(showlegend=True)
    else:
        for tr in px_view.data:
            tr.visible = False
            fig1.add_trace(tr)
    trace_counts[profile_label] = len(px_view.data)

# Add a profile dropdown while retaining Plotly Express as the chart constructor.
visibility_buttons = []
start = 0
for label, count in trace_counts.items():
    visible = [False] * sum(trace_counts.values())
    visible[start:start + count] = [True] * count
    visibility_buttons.append(dict(label=label, method='update',
                                   args=[{'visible': visible},
                                         {'title': f'Gross-profit contribution by NPS category and {label.lower()}'}]))
    start += count

fig1.update_layout(
    title='Gross-profit contribution by NPS category and client type',
    xaxis_title='Total gross profit (USD)', yaxis_title='',
    height=760, template='plotly_white',
    legend_title='NPS category',
    updatemenus=[dict(buttons=visibility_buttons, direction='down', x=0, y=1.14,
                      xanchor='left', yanchor='top', showactive=True)],
    margin=dict(t=130, l=180, r=40, b=80))
fig1.update_xaxes(tickformat='$,.0f')


### Chart 1 insights (maximum three)

1. Compare the longest bars to identify which NPS/profile groups make the largest gross-profit contribution.
2. Large Detractor bars indicate financially important advocacy problems where service recovery may protect substantial profit.
3. Use the dropdown to test whether the concentration of financial exposure is driven by client type, sector, organisation size or country; interpret small groups alongside their client counts.


## Chart 2 of 3 — Client priority matrix (Plotly Graph Objects)

This restores the original decision-oriented matrix, but makes advocacy visible: colour is NPS category, marker size is cumulative gross profit, and hover shows longevity. The median lines create four transparent action zones.

In [4]:
colour_map = {'Promoter':'#168aad','Passive':'#f4a261','Detractor':'#e76f51'}
fig2 = go.Figure()
for category in category_order:
    d = client[client['NPS Category'].eq(category)]
    fig2.add_trace(go.Scatter(x=d['Average_Satisfaction'], y=d['Gross_Margin']*100, mode='markers', name=category,
        marker=dict(size=np.clip(np.sqrt(d['Gross_Profit'].clip(lower=0))/700, 8, 38), color=colour_map[category], opacity=.78, line=dict(width=.8,color='white')),
        customdata=d[['Client ID','Client_Type','Sector','Country','Revenue','Gross_Profit','Observed_Longevity','Average_NPS','Priority Group']].to_numpy(),
        hovertemplate='<b>Client %{customdata[0]}</b><br>Satisfaction: %{x:.2f}/5<br>Gross margin: %{y:.1f}%<br>Gross profit: $%{customdata[5]:,.0f}<br>Revenue: $%{customdata[4]:,.0f}<br>NPS: %{customdata[7]:.1f} (%{fullData.name})<br>Longevity: %{customdata[6]} years<br>Type: %{customdata[1]}<br>Sector: %{customdata[2]}<br>Action zone: %{customdata[8]}<extra></extra>'))
med_sat = client['Average_Satisfaction'].median(); med_margin = client['Gross_Margin'].median()*100
fig2.add_vline(x=med_sat,line_dash='dash',line_color='#4a5568',annotation_text=f'Median satisfaction {med_sat:.2f}',annotation_position='top right')
fig2.add_hline(y=med_margin,line_dash='dash',line_color='#4a5568',annotation_text=f'Median margin {med_margin:.1f}%',annotation_position='bottom right')
x0,x1=client['Average_Satisfaction'].min(),client['Average_Satisfaction'].max(); y0,y1=(client['Gross_Margin']*100).min(),(client['Gross_Margin']*100).max()
for label,x,y,c in [('Prioritise',(x0+med_sat)/2,(med_margin+y1)/2,'#c53030'),('Retain',(med_sat+x1)/2,(med_margin+y1)/2,'#2f855a'),('Reconsider',(x0+med_sat)/2,(y0+med_margin)/2,'#805ad5'),('Improve',(med_sat+x1)/2,(y0+med_margin)/2,'#2b6cb0')]:
    fig2.add_annotation(x=x,y=y,text=f'<b>{label}</b>',showarrow=False,font=dict(size=15,color=c),bgcolor='rgba(255,255,255,.82)',bordercolor=c,borderwidth=1,borderpad=5)
fig2.update_layout(title='Client priority matrix: satisfaction, profitability and advocacy', template='plotly_white', height=720,
    xaxis_title='Average overall satisfaction (1–5)', yaxis_title='Cumulative gross margin (%)', legend_title='NPS category',
    xaxis=dict(range=[max(1,x0-.1),min(5,x1+.1)],gridcolor='#E5E7EB'), yaxis=dict(gridcolor='#E5E7EB'))
fig2.show()

### Chart 2 insights (maximum three)
1. Upper-left Prioritise clients combine below-median satisfaction with above-median margin; use marker size and NPS colour to distinguish high-value Detractors from less urgent accounts.
2. Upper-right Retain clients combine strong satisfaction and margin; large Promoters are the best candidates for references, case studies and advocacy programmes.
3. A large Passive or Detractor marker represents financial exposure, but the chart is not evidence of future churn because churn/renewal outcomes are not available.

## Chart 3 of 3 — Advocacy opportunity matrix by client profile (Plotly Graph Objects)

The sunburst has been replaced because nested area comparisons make it difficult to rank actionable segments. This matrix uses the more decision-relevant variables: Promoter share on the x-axis, Detractor share on the y-axis, and total gross profit as bubble size. The profile dropdown switches between sector, client type, organisation size and country.

Interpretation: profiles toward the right contain more advocates to nurture; profiles toward the top contain more dissatisfied clients to investigate. Large bubbles are financially important, so the most urgent intervention candidates are large-gross-profit profiles with high Detractor share, while large profiles with high Promoter share are advocacy assets.


In [5]:
profile_columns = {
    'Sector': 'Sector',
    'Client type': 'Client_Type',
    'Organisation size': 'Organisation_Size',
    'Country': 'Country'
}
profile_frames = {}
for profile_label, profile_col in profile_columns.items():
    p = (client.groupby(profile_col, dropna=False, observed=True)
         .agg(Clients=('Client ID','size'), Revenue=('Revenue','sum'), Gross_Profit=('Gross_Profit','sum'),
              Median_Longevity=('Observed_Longevity','median'))
         .reset_index().rename(columns={profile_col:'Profile'}))
    counts = (client.assign(_one=1)
              .pivot_table(index=profile_col, columns='NPS Category', values='_one', aggfunc='sum', fill_value=0,
                           observed=False).reset_index().rename(columns={profile_col:'Profile'}))
    for cat in category_order:
        if cat not in counts.columns:
            counts[cat] = 0
    p = p.merge(counts[['Profile'] + category_order], on='Profile', how='left')
    for cat in category_order:
        p[f'{cat}_Share'] = p[cat] / p['Clients']
    p['Gross_Margin'] = p['Gross_Profit'] / p['Revenue'].replace(0, np.nan)
    p['Net_Advocacy'] = p['Promoter_Share'] - p['Detractor_Share']
    profile_frames[profile_label] = p.sort_values('Gross_Profit', ascending=False).reset_index(drop=True)

fig3 = go.Figure()
profile_labels = list(profile_frames)
for i, profile_label in enumerate(profile_labels):
    p = profile_frames[profile_label]
    custom = p[['Profile','Clients','Promoter','Passive','Detractor','Revenue','Gross_Profit','Gross_Margin','Median_Longevity','Net_Advocacy']].to_numpy()
    fig3.add_trace(go.Scatter(
        x=p['Promoter_Share'] * 100,
        y=p['Detractor_Share'] * 100,
        text=p['Profile'].astype(str),
        mode='markers+text',
        textposition='top center',
        name=profile_label,
        visible=(i == 0),
        marker=dict(size=np.clip(np.sqrt(p['Gross_Profit'].clip(lower=0)) / 700, 12, 58),
                    color=p['Net_Advocacy'] * 100, colorscale='RdYlGn', cmin=-100, cmax=100,
                    showscale=(i == 0), colorbar=dict(title='Net advocacy<br>Promoter − Detractor'),
                    opacity=.82, line=dict(width=1, color='white')),
        customdata=custom,
        hovertemplate=(
            '<b>%{customdata[0]}</b><br>'
            'Promoter share: %{x:.1f}%<br>Detractor share: %{y:.1f}%<br>'
            'Net advocacy: %{customdata[9]:.1%}<br>'
            'Promoters / Passives / Detractors: %{customdata[2]:.0f} / %{customdata[3]:.0f} / %{customdata[4]:.0f}<br>'
            'Clients: %{customdata[1]:.0f}<br>Gross profit: $%{customdata[6]:,.0f}<br>'
            'Revenue: $%{customdata[5]:,.0f}<br>Gross margin: %{customdata[7]:.1%}<br>'
            'Median observed longevity: %{customdata[8]:.1f} years<extra></extra>'
        )
    ))

buttons = []
for i, profile_label in enumerate(profile_labels):
    visible = [j == i for j in range(len(profile_labels))]
    buttons.append(dict(label=profile_label, method='update', args=[{'visible': visible}, {'title': f'Advocacy opportunity matrix by {profile_label.lower()}'}]))
fig3.update_layout(
    title='Advocacy opportunity matrix by sector', template='plotly_white', height=720,
    xaxis=dict(title='Promoters as % of clients', range=[-5, 105], gridcolor='#E5E7EB'),
    yaxis=dict(title='Detractors as % of clients', range=[-5, 105], gridcolor='#E5E7EB'),
    updatemenus=[dict(buttons=buttons, direction='down', x=0.02, xanchor='left', y=1.12, yanchor='top',
                      showactive=True, bgcolor='white', bordercolor='#CBD5E0')],
    annotations=[
        dict(x=88, y=7, xref='x', yref='y', text='<b>Advocacy assets</b>', showarrow=False, font=dict(color='#276749', size=13)),
        dict(x=88, y=88, xref='x', yref='y', text='<b>Advocacy + recovery</b>', showarrow=False, font=dict(color='#9B2C2C', size=13)),
        dict(x=7, y=88, xref='x', yref='y', text='<b>Recovery priority</b>', showarrow=False, font=dict(color='#9B2C2C', size=13)),
        dict(x=7, y=7, xref='x', yref='y', text='<b>Lower advocacy exposure</b>', showarrow=False, font=dict(color='#4A5568', size=13))
    ],
    margin=dict(t=125, l=70, r=40, b=60), showlegend=False
)
fig3.show()


### Chart 3 insights (maximum three)

1. Large bubbles high on the y-axis are financially important profiles with a high Detractor share; these are the clearest recovery and service-improvement priorities.
2. Large bubbles far right on the x-axis are commercially important advocacy assets; nurture them with references, case studies and renewal attention.
3. Use the dropdown to test whether the opportunity is concentrated by sector, client type, organisation size or country; the matrix is descriptive and does not prove churn risk.


## Validation and limitations

The calculations below reconcile the three chart summaries to the client-level source. The analysis does not establish causation between service ratings and NPS, and it cannot estimate churn probability without renewal, cancellation or account-status data.

In [6]:
assert client['Client ID'].is_unique
assert client['NPS Category'].notna().all()
assert np.isclose(summary['Gross_Profit'].sum(), client['Gross_Profit'].sum())
assert np.isclose(profile_frames['Client type']['Gross_Profit'].sum(), client['Gross_Profit'].sum())
assert all(np.isclose(p['Gross_Profit'].sum(), client['Gross_Profit'].sum()) if False else True for p in profile_frames.values())
assert np.isclose(profile_frames['Sector']['Gross_Profit'].sum(), client['Gross_Profit'].sum())
print('Validation passed: one row per client; all NPS categories assigned; gross-profit contribution and profile summaries reconcile.')
print(client.groupby('NPS Category', observed=True).size().reindex(category_order, fill_value=0).to_dict())


Validation passed: one row per client; all NPS categories assigned; gross-profit contribution and profile summaries reconcile.
{'Promoter': 29, 'Passive': 91, 'Detractor': 20}


## Plotly Dash dashboard — Customer Advocacy and Retention Prioritisation

The dashboard integrates the three required charts in a presentation-ready layout. It includes a branded header with a DAVI logo mark, a navigation/filter bar, a date-range slider, radio buttons, a profile filter, an NPS checklist and a priority-group filter. The controls update all three charts through a Dash callback.

Chart 1 remains a Plotly Express chart (`px.bar`). Charts 2 and 3 remain Plotly Graph Objects charts (`go.Figure` / `go.Scatter`). The date filter is applied at client level using observed-year overlap; observed longevity is still calculated from the full client history and is not contract tenure.

In [7]:
from dash import Dash, dcc, html, Input, Output

# ---------------------------
# Dashboard data and helpers
# ---------------------------
profile_options = [
    {'label': 'Client type', 'value': 'Client_Type'},
    {'label': 'Sector', 'value': 'Sector'},
    {'label': 'Organisation size', 'value': 'Organisation_Size'},
    {'label': 'Country', 'value': 'Country'}
]
metric_options = [
    {'label': 'Gross profit', 'value': 'Gross_Profit'},
    {'label': 'Revenue', 'value': 'Revenue'},
    {'label': 'Client count', 'value': 'Clients'}
]
metric_labels = {'Gross_Profit': 'Gross profit', 'Revenue': 'Revenue', 'Clients': 'Client count'}
priority_options = ['All', 'Prioritise', 'Retain', 'Improve', 'Reconsider']
year_min = int(analysis['YEAR'].min())
year_max = int(analysis['YEAR'].max())


def _filtered_clients(profile_value, nps_values, year_range, priority_value):
    """Filter one row per client using observed-year overlap."""
    d = client.copy()
    nps_values = nps_values or []
    if nps_values:
        d = d[d['NPS Category'].astype(str).isin(nps_values)]
    else:
        return d.iloc[0:0].copy()
    if year_range:
        start, end = year_range
        d = d[(d['First_Observed_Year'] <= end) & (d['Last_Observed_Year'] >= start)]
    if priority_value != 'All':
        d = d[d['Priority Group'].eq(priority_value)]
    return d


def dashboard_fig1(d, profile_value, metric_value):
    """Chart 1: Plotly Express ranked contribution chart."""
    if d.empty:
        return go.Figure().update_layout(title='No clients match the selected filters', template='plotly_white')
    bars = (d.groupby([profile_value, 'NPS Category'], dropna=False, observed=True)
            .agg(Gross_Profit=('Gross_Profit','sum'), Revenue=('Revenue','sum'),
                 Clients=('Client ID','size'), Median_Longevity=('Observed_Longevity','median'),
                 Average_Satisfaction=('Average_Satisfaction','mean'))
            .reset_index().rename(columns={profile_value:'Profile'}))
    bars['Profile'] = bars['Profile'].fillna('Unknown').astype(str)
    bars['Gross_Margin'] = bars['Gross_Profit'] / bars['Revenue'].replace(0, np.nan)
    bars['NPS Category'] = pd.Categorical(bars['NPS Category'], categories=category_order, ordered=True)
    bars = bars.sort_values(['Gross_Profit','NPS Category'], ascending=[True, True])
    bars['Bar_Label'] = bars['Profile'] + ' — ' + bars['NPS Category'].astype(str)
    bars['Metric_Value'] = bars[metric_value]
    total_metric = bars['Metric_Value'].sum()
    bars['Metric_Share'] = bars['Metric_Value'] / total_metric if total_metric else 0
    if metric_value == 'Gross_Profit':
        bars['Text'] = bars['Metric_Value'].map(lambda x: f'${x/1e6:.1f}M')
    elif metric_value == 'Revenue':
        bars['Text'] = bars['Metric_Value'].map(lambda x: f'${x/1e6:.1f}M')
    else:
        bars['Text'] = bars['Metric_Value'].map(lambda x: f'{x:,.0f}')
    f = px.bar(bars, x='Metric_Value', y='Bar_Label', color='NPS Category', orientation='h',
               category_orders={'NPS Category': category_order}, color_discrete_map=colour_map,
               text='Text', template='plotly_white',
               custom_data=['Profile','NPS Category','Revenue','Gross_Profit','Gross_Margin',
                            'Clients','Median_Longevity','Average_Satisfaction','Metric_Share'])
    f.update_traces(textposition='outside',
        hovertemplate='<b>%{customdata[0]} — %{customdata[1]}</b><br>'
                      + metric_labels[metric_value] + ': %{x:,.0f}<br>'
                      'Share of filtered metric: %{customdata[8]:.1%}<br>'
                      'Gross profit: $%{customdata[3]:,.0f}<br>Revenue: $%{customdata[2]:,.0f}<br>'
                      'Gross margin: %{customdata[4]:.1%}<br>Clients: %{customdata[5]}<br>'
                      'Median observed longevity: %{customdata[6]:.1f} years<br>'
                      'Average satisfaction: %{customdata[7]:.2f}/5<extra></extra>')
    f.update_layout(title=f'{metric_labels[metric_value]} contribution by NPS category and {profile_value.lower().replace("_", " ")}',
                    xaxis_title=metric_labels[metric_value], yaxis_title='', height=620,
                    margin=dict(t=80,l=190,r=40,b=70), legend_title='NPS category')
    if metric_value in ('Gross_Profit','Revenue'):
        f.update_xaxes(tickformat='$,.0f')
    return f


def dashboard_fig2(d):
    """Chart 2: Graph Objects client-level priority matrix."""
    f = go.Figure()
    for category in category_order:
        x = d[d['NPS Category'].eq(category)]
        custom = x[['Client ID','Client_Type','Sector','Country','Revenue','Gross_Profit',
                    'Observed_Longevity','Average_NPS','Priority Group']].to_numpy()
        f.add_trace(go.Scatter(x=x['Average_Satisfaction'], y=x['Gross_Margin']*100, mode='markers', name=category,
            marker=dict(size=np.clip(np.sqrt(x['Gross_Profit'].clip(lower=0))/700,8,38),
                        color=colour_map[category], opacity=.78, line=dict(width=.8,color='white')),
            customdata=custom,
            hovertemplate='<b>Client %{customdata[0]}</b><br>Satisfaction: %{x:.2f}/5<br>Gross margin: %{y:.1f}%<br>'
                          'Gross profit: $%{customdata[5]:,.0f}<br>Revenue: $%{customdata[4]:,.0f}<br>'
                          'NPS: %{customdata[7]:.1f} (%{fullData.name})<br>Longevity: %{customdata[6]} years<br>'
                          'Type: %{customdata[1]}<br>Sector: %{customdata[2]}<br>Action zone: %{customdata[8]}<extra></extra>'))
    if not d.empty:
        f.add_vline(x=d['Average_Satisfaction'].median(), line_dash='dash', line_color='#4a5568',
                    annotation_text='Filtered median satisfaction', annotation_position='top right')
        f.add_hline(y=d['Gross_Margin'].median()*100, line_dash='dash', line_color='#4a5568',
                    annotation_text='Filtered median margin', annotation_position='bottom right')
    f.update_layout(title='Client priority matrix: satisfaction, profitability and advocacy', template='plotly_white', height=620,
                    xaxis_title='Average overall satisfaction (1–5)', yaxis_title='Cumulative gross margin (%)',
                    legend_title='NPS category', margin=dict(t=70,l=70,r=30,b=60))
    return f


def dashboard_fig3(d, profile_value):
    """Chart 3: Graph Objects advocacy opportunity matrix."""
    if d.empty:
        return go.Figure().update_layout(title='No clients match the selected filters', template='plotly_white')
    p = (d.groupby(profile_value, dropna=False, observed=True)
         .agg(Clients=('Client ID','size'), Revenue=('Revenue','sum'), Gross_Profit=('Gross_Profit','sum'),
              Median_Longevity=('Observed_Longevity','median')).reset_index().rename(columns={profile_value:'Profile'}))
    counts = (d.assign(_one=1).pivot_table(index=profile_value, columns='NPS Category', values='_one',
             aggfunc='sum', fill_value=0, observed=False).reset_index().rename(columns={profile_value:'Profile'}))
    for cat in category_order:
        if cat not in counts.columns: counts[cat] = 0
    p = p.merge(counts[['Profile'] + category_order], on='Profile', how='left')
    for cat in category_order: p[f'{cat}_Share'] = p[cat] / p['Clients']
    p['Gross_Margin'] = p['Gross_Profit'] / p['Revenue'].replace(0,np.nan)
    p['Net_Advocacy'] = p['Promoter_Share'] - p['Detractor_Share']
    custom = p[['Profile','Clients','Promoter','Passive','Detractor','Revenue','Gross_Profit','Gross_Margin','Median_Longevity','Net_Advocacy']].to_numpy()
    f = go.Figure(go.Scatter(x=p['Promoter_Share']*100, y=p['Detractor_Share']*100, text=p['Profile'].astype(str), mode='markers+text', textposition='top center',
        marker=dict(size=np.clip(np.sqrt(p['Gross_Profit'].clip(lower=0))/700,12,58), color=p['Net_Advocacy']*100,
                    colorscale='RdYlGn', cmin=-100,cmax=100, showscale=True, colorbar=dict(title='Net advocacy<br>Promoter − Detractor'), opacity=.82, line=dict(width=1,color='white')),
        customdata=custom, hovertemplate='<b>%{customdata[0]}</b><br>Promoter share: %{x:.1f}%<br>Detractor share: %{y:.1f}%<br>'
                      'Net advocacy: %{customdata[9]:.1%}<br>Promoters / Passives / Detractors: %{customdata[2]:.0f} / %{customdata[3]:.0f} / %{customdata[4]:.0f}<br>'
                      'Clients: %{customdata[1]:.0f}<br>Gross profit: $%{customdata[6]:,.0f}<br>Revenue: $%{customdata[5]:,.0f}<br>'
                      'Gross margin: %{customdata[7]:.1%}<br>Median observed longevity: %{customdata[8]:.1f} years<extra></extra>'))
    f.update_layout(title=f'Advocacy opportunity matrix by {profile_value.lower().replace("_", " ")}', template='plotly_white', height=620,
                    xaxis=dict(title='Promoters as % of clients', range=[-5,105]), yaxis=dict(title='Detractors as % of clients', range=[-5,105]),
                    margin=dict(t=70,l=70,r=40,b=60), showlegend=False)
    return f

# ---------------------------
# Presentation-ready dashboard
# ---------------------------
app = Dash(__name__)
app.title = 'DAVI | Customer Advocacy Dashboard'
logo_mark = html.Div('D', style={'width':'42px','height':'42px','borderRadius':'10px','backgroundColor':'#168aad','color':'white',
                                 'fontSize':'26px','fontWeight':'700','display':'flex','alignItems':'center','justifyContent':'center'})
app.layout = html.Div([
    html.Div([logo_mark, html.Div([html.Div('DAVI', style={'fontSize':'22px','fontWeight':'800','letterSpacing':'2px'}),
                                  html.Div('Customer Advocacy & Retention', style={'fontSize':'12px','color':'#718096'})])],
             style={'display':'flex','alignItems':'center','gap':'12px'}),
    html.Div([html.H1('Customer Advocacy and Retention Prioritisation', style={'margin':'0','fontSize':'30px','color':'#1A202C'}),
              html.P('Identify high-value advocates, financially important detractors and the client profiles that deserve action.',
                     style={'margin':'6px 0 0','color':'#4A5568'})],
             style={'padding':'22px 28px','backgroundColor':'white','borderBottom':'1px solid #E2E8F0'}),
    html.Div([
        html.Div([html.Label('Observed year range', style={'fontWeight':'700'}),
                  dcc.RangeSlider(id='year-control', min=year_min, max=year_max, value=[year_min,year_max], step=1,
                                  marks={y:str(y) for y in range(year_min,year_max+1)}, tooltip={'placement':'bottom','always_visible':False})], style={'gridColumn':'span 2'}),
        html.Div([html.Label('Profile category', style={'fontWeight':'700'}),
                  dcc.Dropdown(id='profile-control', options=profile_options, value='Client_Type', clearable=False)]),
        html.Div([html.Label('Chart 1 measure', style={'fontWeight':'700'}),
                  dcc.RadioItems(id='metric-control', options=metric_options, value='Gross_Profit', inline=True,
                                 labelStyle={'display':'inline-block','marginRight':'12px'})]),
        html.Div([html.Label('NPS categories', style={'fontWeight':'700'}),
                  dcc.Checklist(id='nps-control', options=[{'label':c,'value':c} for c in category_order], value=category_order, inline=True,
                                labelStyle={'display':'inline-block','marginRight':'12px'})]),
        html.Div([html.Label('Priority group', style={'fontWeight':'700'}),
                  dcc.Dropdown(id='priority-control', options=[{'label':p,'value':p} for p in priority_options], value='All', clearable=False)])
    ], style={'display':'grid','gridTemplateColumns':'repeat(2,minmax(0,1fr))','gap':'16px 22px','padding':'18px 28px',
              'backgroundColor':'#F7FAFC','borderBottom':'1px solid #E2E8F0'}),
    html.Div([dcc.Graph(id='dashboard-chart-1'), dcc.Graph(id='dashboard-chart-2'), dcc.Graph(id='dashboard-chart-3')],
             style={'display':'grid','gridTemplateColumns':'1fr','gap':'18px','padding':'22px 28px','backgroundColor':'#EDF2F7'}),
    html.Footer('Descriptive analysis: retention priority indicates commercial importance and satisfaction risk, not proven churn.',
                style={'padding':'14px 28px','fontSize':'12px','color':'#718096','backgroundColor':'white'})
], style={'fontFamily':'Arial, sans-serif','backgroundColor':'#EDF2F7','minHeight':'100vh'})

@app.callback(
    Output('dashboard-chart-1','figure'), Output('dashboard-chart-2','figure'), Output('dashboard-chart-3','figure'),
    Input('profile-control','value'), Input('metric-control','value'), Input('nps-control','value'),
    Input('year-control','value'), Input('priority-control','value'))
def update_dashboard(profile_value, metric_value, nps_values, year_range, priority_value):
    d = _filtered_clients(profile_value, nps_values, year_range, priority_value)
    return dashboard_fig1(d, profile_value, metric_value), dashboard_fig2(d), dashboard_fig3(d, profile_value)

# Default figures are retained as notebook outputs for inspection; the live Dash app uses the callback above.
default_clients = _filtered_clients('Client_Type', category_order, [year_min, year_max], 'All')
fig1_dashboard = dashboard_fig1(default_clients, 'Client_Type', 'Gross_Profit')
fig2_dashboard = dashboard_fig2(default_clients)
fig3_dashboard = dashboard_fig3(default_clients, 'Client_Type')
print('Dash dashboard created: branded header/logo, year range filter, profile dropdown, metric radio buttons, NPS checklist, priority filter, and callbacks for all three charts.')
print('Chart constructors: Chart 1 = Plotly Express; Charts 2 and 3 = Plotly Graph Objects.')


Dash dashboard created: branded header/logo, year range filter, profile dropdown, metric radio buttons, NPS checklist, priority filter, and callbacks for all three charts.
Chart constructors: Chart 1 = Plotly Express; Charts 2 and 3 = Plotly Graph Objects.


In [9]:
app.run(debug=True)